# Fit and calibrate a router package

This NB-004 workflow consumes an immutable NB-003 expert-only package, fits CAL-004 utility heads, applies CAL-005 temperature and threshold calibration, and derives a new router-bearing package version without modifying its source.

## Resolve configuration

The resolved mapping keeps fitting, calibration, package versioning, and runtime choices explicit.

In [ ]:
import os
import platform
from pathlib import Path
from tempfile import TemporaryDirectory

import accelerate
import safetensors
import torch
import transformers
from transformers import Qwen3Config, Qwen3ForCausalLM

from anchored_moe import (
    AnchoredMoEConfig,
    AnchoredTop1Selection,
    CalibratedRouterPackageConfig,
    ExpertPackageExportBinding,
    ExpertPackageExportConfig,
    ExpertRegistry,
    FullMLPExpert,
    RouterFeatureBank,
    UtilityLabelBank,
    calibrate_temperature_and_thresholds,
    derive_calibrated_router_package,
    export_full_mlp_expert_package,
    fit_independent_ridge,
    synchronize_anchored_top1_registry,
)
from anchored_moe.backends import Qwen3Backend
from anchored_moe.evaluation import fingerprint_module
from anchored_moe.utils import sha256_json

smoke_test = os.environ.get("ANCHORED_MOE_SMOKE_TEST", "1") == "1"
resolved_config = {
    "feature_id": "NB-004",
    "base_model": {
        "model_id": "local-random-qwen3",
        "revision": "synthetic-seed-13",
        "converted_layers": [2, 3],
        "dtype": "float32",
    },
    "source_package": {
        "specialization_name": "code",
        "specialization_version": "1.0.0",
        "expert_id": "code:0",
    },
    "fitting": {
        "strategy": "independent_ridge",
        "regularization": 1e-6,
    },
    "calibration": {
        "strategy": "temperature_threshold",
        "dataset_version": "synthetic-router-v1",
        "temperature_row_ids": [10, 11, 12, 13],
        "threshold_row_ids": [14, 15, 16, 17, 18, 19],
        "false_activation_rate": 1.0 / 3.0,
    },
    "derived_package": {
        "specialization_version": "1.0.0+router.1",
        "selection": "anchored_top1",
    },
    "router_features": {
        "feature_type": "identity",
        "input": "pre_mlp_hidden_state",
        "output_size": 16,
        "version": 1,
    },
}
resolved_config

## Load the expert package and calibration data

The offline acceptance path creates the same expert-only artifact contract as NB-003, then constructs distinct fitting and held-out feature/utility banks with globally disjoint row IDs.

In [ ]:
torch.manual_seed(13)
model = Qwen3ForCausalLM(
    Qwen3Config(
        vocab_size=64,
        hidden_size=16,
        intermediate_size=32,
        num_hidden_layers=4,
        num_attention_heads=4,
        num_key_value_heads=2,
        head_dim=4,
        max_position_embeddings=32,
    )
).eval()
runtime_versions = {
    "accelerate": accelerate.__version__,
    "kernels": "torch-default-cpu",
    "peft": "not-installed",
    "python": platform.python_version(),
    "safetensors": safetensors.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
}
temporary_workspace = TemporaryDirectory(prefix="anchored-moe-nb004-")
workspace = Path(temporary_workspace.name)
source_package = export_full_mlp_expert_package(
    workspace / "code-expert-only",
    bindings=[
        ExpertPackageExportBinding(
            layer_index,
            resolved_config["source_package"]["expert_id"],
            FullMLPExpert.from_anchor(model.model.layers[layer_index].mlp),
        )
        for layer_index in resolved_config["base_model"]["converted_layers"]
    ],
    config=ExpertPackageExportConfig(
        specialization_name=resolved_config["source_package"]["specialization_name"],
        specialization_version=resolved_config["source_package"]["specialization_version"],
        base_model_repository=resolved_config["base_model"]["model_id"],
        base_model_revision=resolved_config["base_model"]["revision"],
        base_config_sha256=sha256_json(model.config.to_dict()),
        base_weight_sha256={"in-memory-anchor-state": fingerprint_module(model)},
        tokenizer_repository=resolved_config["base_model"]["model_id"],
        tokenizer_revision=resolved_config["base_model"]["revision"],
        base_model_dtype=resolved_config["base_model"]["dtype"],
        runtime_versions=runtime_versions,
        training_dataset={
            "dataset_id": "synthetic-local-code",
            "revision": "1",
            "split": "train",
            "filters": [],
        },
        router_feature_specification=resolved_config["router_features"],
        expert_parameterization_config={
            "initialization": "anchor_copy",
            "training_strategy": "forced_path",
        },
    ),
)
source_files_before = {
    path.relative_to(source_package.package_directory).as_posix(): path.read_bytes()
    for path in source_package.package_directory.rglob("*")
    if path.is_file()
}

In [ ]:
fitting_scalars = torch.tensor([-3.0, -2.0, -1.0, 1.0, 2.0, 3.0])
fitting_features = torch.zeros(len(fitting_scalars), 16)
fitting_features[:, 0] = fitting_scalars
fitting_bank = RouterFeatureBank(
    {2: fitting_features, 3: fitting_features.clone()},
    feature_specification=resolved_config["router_features"],
    dataset_provenance={
        "dataset_id": "synthetic-local-router",
        "revision": "1",
        "split": "train",
        "filters": [],
    },
    seed=73,
    row_ids=torch.arange(6),
)
fitting_values = fitting_scalars.reshape(-1, 1)
fitting_labels = UtilityLabelBank(
    {2: fitting_values, 3: fitting_values.clone()},
    expert_ids_by_layer={2: ("code:0",), 3: ("code:0",)},
    row_ids=fitting_bank.row_ids,
    feature_bank_metadata_sha256=fitting_bank.metadata_sha256,
    loss_name="cross_entropy",
    incumbent_route="anchor",
)

calibration_scalars = torch.tensor(
    [-2.0, -1.0, 1.0, 2.0, -0.5, 0.5, 1.0, 2.0, 3.0, 4.0]
)
calibration_utilities = torch.tensor(
    [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
)
calibration_features = torch.zeros(len(calibration_scalars), 16)
calibration_features[:, 0] = calibration_scalars
calibration_bank = RouterFeatureBank(
    {2: calibration_features, 3: calibration_features.clone()},
    feature_specification=resolved_config["router_features"],
    dataset_provenance={
        "dataset_id": "synthetic-local-router",
        "revision": "1",
        "split": "validation",
        "filters": [],
    },
    seed=73,
    row_ids=torch.arange(10, 20),
)
calibration_values = calibration_utilities.reshape(-1, 1)
calibration_labels = UtilityLabelBank(
    {2: calibration_values, 3: calibration_values.clone()},
    expert_ids_by_layer={2: ("code:0",), 3: ("code:0",)},
    row_ids=calibration_bank.row_ids,
    feature_bank_metadata_sha256=calibration_bank.metadata_sha256,
    loss_name="cross_entropy",
    incumbent_route="anchor",
)

## Run framework operation

Fit frozen independent heads, calibrate them on explicit disjoint partitions, save the reproducibility banks, and derive a new immutable package version.

In [ ]:
fitted_router = fit_independent_ridge(
    fitting_bank,
    fitting_labels,
    regularization=resolved_config["fitting"]["regularization"],
)
calibrated_router, calibration_report = calibrate_temperature_and_thresholds(
    fitted_router,
    calibration_bank,
    calibration_labels,
    temperature_row_ids=resolved_config["calibration"]["temperature_row_ids"],
    threshold_row_ids=resolved_config["calibration"]["threshold_row_ids"],
    false_activation_rate=resolved_config["calibration"]["false_activation_rate"],
)
bank_artifacts = workspace / "router-data"
bank_artifacts.mkdir()
fitting_bank.save_safetensors(bank_artifacts / "fitting-features.safetensors")
fitting_labels.save_safetensors(bank_artifacts / "fitting-utilities.safetensors")
calibration_bank.save_safetensors(bank_artifacts / "calibration-features.safetensors")
calibration_labels.save_safetensors(bank_artifacts / "calibration-utilities.safetensors")

In [ ]:
derived_package = derive_calibrated_router_package(
    source_package,
    workspace / "code-routed",
    calibrated_router=calibrated_router,
    calibration_report=calibration_report,
    fitting_feature_bank=fitting_bank,
    fitting_utility_labels=fitting_labels,
    calibration_feature_bank=calibration_bank,
    calibration_utility_labels=calibration_labels,
    config=CalibratedRouterPackageConfig(
        specialization_version=resolved_config["derived_package"]["specialization_version"],
        calibration_dataset_version=resolved_config["calibration"]["dataset_version"],
        ridge_regularization=resolved_config["fitting"]["regularization"],
        runtime_versions=runtime_versions,
    ),
)
derived_package.identity

## Validate invariants and summarize

The checks cover source immutability, anchor and expert routes, frozen calibrated heads, strict package validation, and model-level materialization.

In [ ]:
source_files_after = {
    path.relative_to(source_package.package_directory).as_posix(): path.read_bytes()
    for path in source_package.package_directory.rglob("*")
    if path.is_file()
}
scores = calibrated_router(
    calibration_bank.features_for_layer(2),
    layer_index=2,
)
decision = AnchoredTop1Selection()(
    scores,
    expert_ids=calibrated_router.expert_ids_for_layer(2),
)
route_ids = decision.selected_indices.tolist()
assert source_files_after == source_files_before
assert 0 in route_ids and 1 in route_ids
assert all(not parameter.requires_grad for parameter in calibrated_router.parameters())
assert derived_package.manifest.router_architecture == "independent_affine_heads"
derived_package.validate_live_files()

backend = Qwen3Backend()
backend.convert(
    model,
    AnchoredMoEConfig(converted_layers=resolved_config["base_model"]["converted_layers"]),
)
registry = ExpertRegistry()
registry.install(derived_package)
composition = synchronize_anchored_top1_registry(
    model,
    backend=backend,
    registry=registry,
)
calibration_summary = {
    "feature_id": resolved_config["feature_id"],
    "head_keys": [list(key) for key in calibrated_router.head_keys],
    "temperature_rows": list(calibration_report.temperature_row_ids),
    "threshold_rows": list(calibration_report.threshold_row_ids),
    "route_ids": route_ids,
    "source_immutable": source_files_after == source_files_before,
    "derived_version": derived_package.identity.specialization_version,
    "configured_layers": list(composition.configured_layers),
    "manifest_sha256": derived_package.identity.manifest_sha256,
}
print("NB-004 validation passed", calibration_summary)